In [58]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import statsmodels.api as sm
import warnings

warnings.filterwarnings('ignore')

In [59]:
symbol = 'BITW'
filename = f"{symbol}_weekly_data.csv"

# Load the data
df = pd.read_csv(filename, index_col=0, parse_dates=True)

# Display the first few rows
display(df.head())

# Remove anomalous data points on and before November 27, 2020
cutoff = pd.to_datetime('2020-11-27')
df = df[df.index >= cutoff]

,Open,High,Low,Close,Volume
Date,,,,,
2020-10-23,12.0,12.0,12.0,12.0,0.0
2020-10-30,12.0,12.0,12.0,12.0,0.0
2020-11-06,12.0,12.0,12.0,12.0,0.0
2020-11-13,12.0,12.0,12.0,12.0,0.0
2020-11-20,12.0,12.0,12.0,12.0,112.0


In [60]:
fig = px.line(
    df, 
    x=df.index, 
    y='Close', 
    title=f'{symbol} Historical Weekly Closing Price',
    labels={'index': 'Date', 'Close': 'Price (USD)'}
)

fig.update_layout(template='plotly_dark') # Optional: 'plotly_white' or 'plotly_dark' for a modern feel
fig.show()

In [61]:
# Calculate 10-week and 40-week SMAs
df['SMA_10'] = df['Close'].rolling(window=10).mean()
df['SMA_40'] = df['Close'].rolling(window=40).mean()

# Plot Close, SMA_10, and SMA_40
fig = px.line(
    df, 
    x=df.index, 
    y=['Close', 'SMA_10', 'SMA_40'],
    title=f'{symbol} Price with 10-Week and 40-Week Moving Averages',
    labels={'value': 'Price (USD)', 'variable': 'Legend'}
)

fig.update_layout(template='plotly_dark', hovermode='x unified')
fig.show()

In [62]:
# Calculate weekly percentage change
df['Weekly_Return'] = df['Close'].pct_change()

# Plot the distribution of returns as a histogram
fig = px.histogram(
    df, 
    x='Weekly_Return', 
    nbins=50, 
    title=f'Distribution of {symbol} Weekly Returns',
    labels={'Weekly_Return': 'Percentage Return'},
    marginal='box' # Adds a modern boxplot at the top to see outliers
)

fig.update_layout(template='plotly_dark')
fig.show()

# Calculate annualized volatility
annual_volatility = df['Weekly_Return'].std() * np.sqrt(52)
print(f"Annualized Volatility: {annual_volatility:.2%}")

Annualized Volatility: 113.78%


In [63]:
# We will use subplots: Top chart for Candlesticks, Bottom chart for Volume
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, 
                    vertical_spacing=0.03, 
                    row_heights=[0.7, 0.3])

# 1. Add Candlestick
fig.add_trace(go.Candlestick(
    x=df.index,
    open=df['Open'],
    high=df['High'],
    low=df['Low'],
    close=df['Close'],
    name='Price'
), row=1, col=1)

# 2. Add Volume Bar Chart
# Color the volume bars based on whether it was an up or down week
colors = ['green' if row['Close'] >= row['Open'] else 'red' for index, row in df.iterrows()]

fig.add_trace(go.Bar(
    x=df.index, 
    y=df['Volume'],
    marker_color=colors,
    name='Volume'
), row=2, col=1)

# Update layout for a clean, modern look
fig.update_layout(
    title=f'{symbol} Weekly Price & Volume Action',
    template='plotly_dark',
    xaxis_rangeslider_visible=False, # Disable the default rangeslider to keep it clean with the volume subplot
    height=700
)

fig.show()

In [64]:
# Load the Wilshire 5000 data
# Update the path if your notebook is in a different directory relative to the resources folder
wilshire_df = pd.read_csv('./WeeklyWilshire5000.csv')

# Clean the 'Date' column and set it as the index
wilshire_df['Date'] = pd.to_datetime(wilshire_df['Date'])
wilshire_df.set_index('Date', inplace=True)

# Clean the 'Close' column (remove commas and convert to float)
wilshire_df['Close'] = wilshire_df['Close'].astype(str).str.replace(',', '').astype(float)

# Rename the column for clarity before merging
wilshire_df = wilshire_df[['Close']].rename(columns={'Close': 'Wilshire_Close'})

# Isolate the relevant columns from our existing BITW DataFrame ('df')
bitw_subset = df[['Close', 'Weekly_Return']].rename(columns={'Close': 'BITW_Close', 'Weekly_Return': 'BITW_Return'})

# Merge the two datasets on the Date index (inner join ensures we only keep overlapping dates)
merged_df = bitw_subset.join(wilshire_df, how='inner')

# Calculate the Wilshire 5000 weekly returns
merged_df['Wilshire_Return'] = merged_df['Wilshire_Close'].pct_change()

# Drop the NaN values created by the pct_change calculation
merged_df.dropna(inplace=True)

# Calculate and print the correlation coefficients
price_corr = merged_df['BITW_Close'].corr(merged_df['Wilshire_Close'])
return_corr = merged_df['BITW_Return'].corr(merged_df['Wilshire_Return'])

print(f"Overall Price Correlation: {price_corr:.3f}")
print(f"Overall Weekly Returns Correlation: {return_corr:.3f}")

Overall Price Correlation: 0.668
Overall Weekly Returns Correlation: 0.181


In [65]:
# 1. Calculate Cumulative Returns (Growth of $100)
# This simulates investing $100 in each asset at the start of our overlapping time period
merged_df['BITW_Cumulative'] = (1 + merged_df['BITW_Return']).cumprod() * 100
merged_df['Wilshire_Cumulative'] = (1 + merged_df['Wilshire_Return']).cumprod() * 100

# Create a clean line chart for Cumulative Returns
fig1 = px.line(
    merged_df,
    x=merged_df.index,
    y=['BITW_Cumulative', 'Wilshire_Cumulative'],
    title='Comparative Performance: Growth of $100 (BITW vs Wilshire 5000)',
    labels={'value': 'Value (USD)', 'variable': 'Asset', 'index': 'Date'}
)

# Clean up the legend and layout
fig1.update_layout(
    template='plotly_dark', 
    hovermode='x unified',
    legend_title_text=''
)

# Rename the legend traces for readability
newnames = {'BITW_Cumulative': 'BITW', 'Wilshire_Cumulative': 'Wilshire 5000'}
fig1.for_each_trace(lambda t: t.update(name = newnames[t.name],
                                      legendgroup = newnames[t.name],
                                      hovertemplate = t.hovertemplate.replace(t.name, newnames[t.name])))
fig1.show()


# 2. Clean Correlation Matrix (Heatmap)
# Calculate the correlation matrix of the returns
corr_matrix = merged_df[['BITW_Return', 'Wilshire_Return']].corr()

# Rename columns/index for a cleaner look on the chart
corr_matrix.columns = ['BITW Returns', 'Wilshire 5000 Returns']
corr_matrix.index = ['BITW Returns', 'Wilshire 5000 Returns']

fig2 = px.imshow(
    corr_matrix,
    text_auto=".2f", # Show the exact correlation number rounded to 2 decimals
    aspect="auto",
    color_continuous_scale='RdBu_r', # Red (negative) to Blue (positive)
    zmin=-1, zmax=1, # Lock the scale to standard correlation bounds
    title='Correlation Heatmap'
)

fig2.update_layout(template='plotly_dark', width=600, height=500)
fig2.show()

In [66]:
# --- Store of Value Stress Testing ---

# 1. Downside Correlation (When the Stock Market Drops)
# Isolate weeks where the Wilshire 5000 had a negative return
downturn_market = merged_df[merged_df['Wilshire_Return'] < 0]

downside_corr = downturn_market['BITW_Return'].corr(downturn_market['Wilshire_Return'])
print(f"Correlation strictly during Wilshire 5000 Down-Weeks: {downside_corr:.3f}")

# Calculate Average Returns during market down-weeks
avg_wilshire_drop = downturn_market['Wilshire_Return'].mean()
avg_bitw_during_drop = downturn_market['BITW_Return'].mean()

print(f"Average Wilshire Return during down-weeks: {avg_wilshire_drop:.2%}")
print(f"Average BITW Return during those same weeks: {avg_bitw_during_drop:.2%}")


# 2. Drawdown Analysis
# Calculate the running maximum (peak) for both assets
merged_df['BITW_Peak'] = merged_df['BITW_Close'].cummax()
merged_df['Wilshire_Peak'] = merged_df['Wilshire_Close'].cummax()

# Calculate the percentage drop from the peak (Drawdown)
merged_df['BITW_Drawdown'] = (merged_df['BITW_Close'] - merged_df['BITW_Peak']) / merged_df['BITW_Peak']
merged_df['Wilshire_Drawdown'] = (merged_df['Wilshire_Close'] - merged_df['Wilshire_Peak']) / merged_df['Wilshire_Peak']

# Visualize the Drawdowns
fig = px.area(
    merged_df,
    x=merged_df.index,
    y=['BITW_Drawdown', 'Wilshire_Drawdown'],
    title='Drawdown Profile: Depth of Losses from Peak',
    labels={'value': 'Percentage Drop from Peak', 'index': 'Date', 'variable': 'Asset'}
)

# Update layout for readability (format y-axis as percentage)
fig.update_layout(
    template='plotly_dark',
    hovermode='x unified',
    yaxis_tickformat='.0%'
)

# Clean up legend names
newnames = {'BITW_Drawdown': 'BITW Drawdown', 'Wilshire_Drawdown': 'Wilshire 5000 Drawdown'}
fig.for_each_trace(lambda t: t.update(name = newnames[t.name],
                                      legendgroup = newnames[t.name],
                                      hovertemplate = t.hovertemplate.replace(t.name, newnames[t.name])))

fig.show()

Correlation strictly during Wilshire 5000 Down-Weeks: 0.090
Average Wilshire Return during down-weeks: -1.88%
Average BITW Return during those same weeks: -1.82%
